In [ ]:
from torch.utils.data import Dataset
import numpy as np
import shapeworld
import os
import torch
from torch.utils.data import Sampler
from collections import defaultdict
import matplotlib.pyplot as plt
import random

In [ ]:
class MultiShapeWorld():
    def __init__(self, n=100, mode='train', config='classification.json', num_objects=2):
        dataset = shapeworld.Dataset.create(dtype='classification', name='shape', collision_tolerance=0.0, config=config)
        generated = dataset.generate(n=n, mode=mode, include_model=True)
        self.imgs = generated['world']
        self.labels = generated['classification']

        correct_count = self.labels.sum(axis=1) == num_objects
        self.imgs = self.imgs[correct_count]
        self.labels = self.labels[correct_count]
        
        self.single_labels = []
        for i in range(len(self.imgs)):
            classes = np.where(self.labels[i] == 1)[0] 
            quotient, remainder = np.divmod(classes, 7)
            shapes = "".join(map(str, sorted(quotient)))
            colors = "".join(map(str, sorted(remainder)))
            self.single_labels.append((shapes, colors))
            
        self.dataset = []
        for i in range(len(self.imgs)):
            self.dataset.append((self.imgs[i], self.single_labels[i]))
            
    def get_labels(self):
        return self.single_labels
    
    def get_dataset(self):
        return self.dataset

In [ ]:
class CrossLabelBatchSampler(Sampler):
    def __init__(self, labels, batch_size_per_label=2):
        """
        labels: list of tuples [(label1, label2), ...] for the dataset
        batch_size_per_label: B, number of label1s and label2s per batch
        """
        self.labels = labels
        self.batch_size_per_label = batch_size_per_label
        self.label_pair_to_indices = defaultdict(list)
        
        for idx, (l1, l2) in enumerate(labels):
            self.label_pair_to_indices[(l1, l2)].append(idx)
            
        self.all_label1s = list(set(l1 for l1, l2 in labels))
        self.all_label2s = list(set(l2 for l1, l2 in labels))

    def __iter__(self):
        used_indices = set()
        batches = []

        while len(used_indices) < len(self.labels):
            # Available labels for this batch
            available_label1s = [l for l in self.all_label1s if any(
                idx not in used_indices for l2 in self.all_label2s for idx in self.label_pair_to_indices.get((l, l2), [])
            )]
            available_label2s = [l for l in self.all_label2s if any(
                idx not in used_indices for l1 in self.all_label1s for idx in self.label_pair_to_indices.get((l1, l), [])
            )]

            if not available_label1s or not available_label2s:
                break

            # Choose B labels randomly
            chosen_label1s = random.sample(available_label1s, min(self.batch_size_per_label, len(available_label1s)))
            chosen_label2s = random.sample(available_label2s, min(self.batch_size_per_label, len(available_label2s)))

            batch = []

            # Cross-product of chosen labels
            for l1 in chosen_label1s:
                for l2 in chosen_label2s:
                    candidates = [idx for idx in self.label_pair_to_indices.get((l1, l2), []) if idx not in used_indices]
                    if candidates:
                        chosen_idx = random.choice(candidates)
                        batch.append(chosen_idx)
                        used_indices.add(chosen_idx)

            if len(batch) == self.batch_size_per_label ** 2:
                batches.append(batch)
                yield batch

    def __len__(self):
        # Approximate number of batches
        total_samples = len(self.labels)
        batch_size = self.batch_size_per_label ** 2
        return (total_samples + batch_size - 1) // batch_size


In [ ]:
class SingleUniqueLabelSampler(Sampler):
    def __init__(self, attributes, batch_size):
        """
        attributes: list of tuples [(attr1, attr2), ...]
        batch_size: B
        """
        self.attributes = attributes
        self.batch_size = batch_size

        # attr1_value -> [indices]
        self.attr1_to_indices = defaultdict(list)
        # attr2_value -> [indices]
        self.attr2_to_indices = defaultdict(list)

        for idx, (a1, a2) in enumerate(attributes):
            self.attr1_to_indices[a1].append(idx)
            self.attr2_to_indices[a2].append(idx)
            
        print("shape", len(self.attr1_to_indices.keys()))
        print("color", len(self.attr2_to_indices.keys()), self.attr2_to_indices.keys())


    def __iter__(self):
        unused_indices = set(range(len(self.attributes)))

        while True:
            # Stop if not enough samples left
            if len(unused_indices) < self.batch_size:
                break

            # Randomly choose attribute 1 or 2
            chosen_attr = random.choice([1, 2])

            if chosen_attr == 1:
                attr_map = self.attr1_to_indices
                attr_same = self.attr2_to_indices
            else:
                attr_map = self.attr2_to_indices
                attr_same = self.attr1_to_indices

                
            for value, idxs in attr_map.items():
                print(f"value {value}: {len(idxs)} indices")


            attr_same_values = list(attr_same.keys())
            random.shuffle(attr_same_values)
            for attr_same_value in attr_same_values:
                batch = []
                # Shuffle attribute values
                attr_values = list(attr_map.keys())
                random.shuffle(attr_values)

                for attr_val in attr_values:
                    # Available samples with this attribute value
                    candidates = [
                        idx for idx in attr_map[attr_val]
                        if idx in unused_indices and idx in attr_same[attr_same_value]
                    ]

                    if not candidates:
                        continue

                    chosen_idx = random.choice(candidates)
                    batch.append(chosen_idx)
                    unused_indices.remove(chosen_idx)

                    if len(batch) == self.batch_size:
                        yield batch
                        break

            # if len(batch) < self.batch_size:
                # break

            yield batch

    def __len__(self):
        return len(self.attributes) // self.batch_size


In [ ]:
from torch.utils.data import DataLoader

train_dataset = MultiShapeWorld(n=100000, mode='train')
val_dataset = MultiShapeWorld(n=10000, mode='validation')
test_dataset = MultiShapeWorld(n=10000, mode='test')

for (name, dataset, num_batches) in zip(['train', 'validation', 'test'], [train_dataset, val_dataset, test_dataset], [650, 70, 70]):
    sampler = SingleUniqueLabelSampler(
        attributes=dataset.get_labels(),
        batch_size=28,
    )

    loader = DataLoader(
        dataset.get_dataset(),
        batch_sampler=sampler,
        num_workers=0
    )

    saved_batches = []
    for i, batch in enumerate(loader):
        if i >= num_batches:
            break
        saved_batches.append(batch)
        print(i)

    random.shuffle(saved_batches)
    # torch.save(saved_batches, f"/home/shared/data/shape_unique_single_attribute/{name}.pt")
    print(len(saved_batches))

In [ ]:
class PreSavedBatchDataset(Dataset):
    def __init__(self, batches):
        self.batches = batches

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, idx):
        return self.batches[idx]

In [ ]:
train_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute/train.pt"))
val_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute/validation.pt"))
test_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute/test.pt"))


# Use batch_size=1 because each "item" is already a batch
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)


In [ ]:
# Test correcness
for batch in saved_batches:
    idxs = batch[1]
    if len(set(idxs[0])) != 28 and len(set(idxs[1])) != 28:
        print(len(idxs[0]), len(idxs[1]))
        
    assert len(set(idxs[0])) == 28 or len(set(idxs[1])) == 28 or len(set(idxs[0])) == 28

In [ ]:
def plot_images(imgs):
    n = len(imgs)
    cols = 5
    rows = int(np.ceil(n / cols))
    
    plt.figure(figsize=(cols * 3, rows * 3))
    
    for i, img in enumerate(imgs):
        plt.subplot(rows, cols, i + 1)
        
        # Handle grayscale vs RGB
        if img.ndim == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(img)
        
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_images(train_dataset[0][0])

In [ ]:
plot_images(val_dataset[0][0])

In [ ]:
plot_images(test_dataset[0][0])

In [ ]:
test_time_dataset = MultiShapeWorld(n=100000, mode='test')

test_time_sampler = CrossLabelBatchSampler(test_time_dataset.get_labels(), batch_size_per_label=10)

test_time_loader = DataLoader(
    test_time_dataset.get_dataset(),
    batch_sampler=test_time_sampler,
    num_workers=4
)


saved_batches = []
for i, batch in enumerate(test_time_loader):
    if i >= 20:
        break
    saved_batches.append(batch)
len(saved_batches)

random.shuffle(saved_batches)
# torch.save(saved_batches, f"/home/shared/data/shape_unique_double_attribute/test.pt")
